# 📈 Monte Carlo Stock Price Simulator

Simulates thousands of possible future price paths for a stock using **probability and random sampling**.

**Tools:** Python, NumPy, pandas, Matplotlib
**Output:** 1,000+ simulated price paths, scenario distribution and confidence intervals.

👉 Run each cell from top to bottom (click ▶ or press `Shift + Enter`).

## Step 0: Install and import libraries

In [ ]:
!pip install yfinance -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

## Step 1: Settings
Change these values to experiment!

In [ ]:
TICKER = "AAPL"          # Stock symbol (e.g. "TSLA", "MSFT", "RELIANCE.NS")
HISTORY_PERIOD = "2y"    # How much past data to learn from
DAYS = 252               # Days to simulate (252 trading days = 1 year)
SIMULATIONS = 1000       # Number of possible futures to simulate

## Step 2: Get historical stock prices

In [ ]:
try:
    data = yf.download(TICKER, period=HISTORY_PERIOD, progress=False)
    prices = data["Close"].squeeze().dropna()
    if len(prices) < 30:
        raise ValueError("Not enough data")
    print(f"Downloaded {len(prices)} days of {TICKER} prices.")
except Exception as error:
    print(f"Could not download data ({error}). Using sample data instead.")
    np.random.seed(1)
    fake_returns = np.random.normal(0.0005, 0.015, 500)
    prices = pd.Series(100 * np.cumprod(1 + fake_returns))

prices.plot(title=f"{TICKER} Historical Prices", figsize=(8, 4))
plt.show()

## Step 3: Learn from history
We measure two things:
- **Average daily return** – the typical daily move
- **Volatility** – how much the price jumps around

In [ ]:
daily_returns = prices.pct_change().dropna()

average = daily_returns.mean()
volatility = daily_returns.std()
start_price = float(prices.iloc[-1])

print(f"Starting price:       {start_price:.2f}")
print(f"Average daily return: {average:.4%}")
print(f"Daily volatility:     {volatility:.4%}")

## Step 4: Run the Monte Carlo simulation
For each simulation we pick random daily returns, then compound them day after day to build a price path.

In [ ]:
# Random daily returns: one column per simulation
random_returns = np.random.normal(average, volatility, (DAYS, SIMULATIONS))

# Turn returns into prices
paths = start_price * np.cumprod(1 + random_returns, axis=0)

# Add the starting price as the first row
start_row = np.full((1, SIMULATIONS), start_price)
paths = np.vstack([start_row, paths])

print(f"Created {paths.shape[1]} price paths, each {paths.shape[0]} days long.")

## Step 5: Summarize the results

In [ ]:
final_prices = paths[-1]

low = np.percentile(final_prices, 5)
high = np.percentile(final_prices, 95)

print(f"Starting price:               {start_price:.2f}")
print(f"Average final price:          {final_prices.mean():.2f}")
print(f"Median final price:           {np.median(final_prices):.2f}")
print(f"Worst case (5th percentile):  {low:.2f}")
print(f"Best case (95th percentile):  {high:.2f}")
print(f"Chance of profit:             {(final_prices > start_price).mean() * 100:.1f}%")
print(f"\n90% of simulations ended between {low:.2f} and {high:.2f}")

## Step 6: Visualize the results

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: all simulated price paths
left.plot(paths, alpha=0.1, linewidth=0.7)
left.plot(paths.mean(axis=1), color="black", linewidth=2, label="Average path")
left.set_title(f"{SIMULATIONS} Simulated Price Paths for {TICKER}")
left.set_xlabel("Days")
left.set_ylabel("Price")
left.legend()

# Chart 2: distribution of final prices + confidence interval
right.hist(final_prices, bins=50, color="skyblue", edgecolor="white")
right.axvline(start_price, color="black", linestyle="--", label="Start price")
right.axvline(low, color="red", linestyle="--", label="5th percentile")
right.axvline(high, color="green", linestyle="--", label="95th percentile")
right.set_title("Distribution of Final Prices (90% Confidence Interval)")
right.set_xlabel("Final price")
right.set_ylabel("Number of simulations")
right.legend()

plt.tight_layout()
plt.savefig("simulation_results.png", dpi=150)
plt.show()

## Step 7 (optional): Download your results

In [ ]:
pd.DataFrame(paths).to_csv("simulated_paths.csv", index=False)

try:
    from google.colab import files
    files.download("simulation_results.png")
    files.download("simulated_paths.csv")
except ImportError:
    print("Saved simulation_results.png and simulated_paths.csv in this folder.")

---
⚠️ **Limitations:** This is a learning project. It assumes returns are normally distributed and that past average/volatility continue into the future, which real markets don't guarantee. **Not financial advice.**